- https://huggingface.co/docs/trl/main/en/gkd_trainer

### gold

- cross tokenizer
    - Hugging Face 提出的 GOLD (General On-Policy Logit Distillation) 就是为了打破这个限制。
    - 不管老师和学生怎么切分句子，GOLD 先把它们还原成相同的文本字符。
        - 学生说：Hug (token A) + ging (token B)
        - 老师说：Hugging (token C)

> 整个过程分为三步：采样 (Sampling)、对齐 (Alignment) 和 优化 (Optimization)。
- 采样：$\hat{y} \sim \pi_\theta(\cdot | x)$
- 对齐：
    - 学生可能将其切分为 $N$ 个 Token（Tokenizer $S$）：$u = (u_1, u_2, \dots, u_N)$，学生的生成概率是所有 Token 概率的累积：
        - $\log P_\theta(\hat{y} | x) = \sum_{i=1}^{N} \log \pi_\theta(u_i | u_{<i}, x)$
    - 老师可能将其切分为 $M$ 个 Token（Tokenizer $T$）：$v = (v_1, v_2, \dots, v_M)$注意：$N \neq M$，且 Token ID 完全不同。为了评估学生的回答，我们将文本 $\hat{y}$ 强制输入给老师模型，计算老师生成同一段文本的概率：
        - $\log P_\phi(\hat{y} | x) = \sum_{j=1}^{M} \log \pi_\phi(v_j | v_{<j}, x)$
    - 虽然 Token 序列 $u$ 和 $v$ 无法一一对应，但它们的序列总对数似然 (Total Log-Likelihood) 是标量，是可以直接比较的。
- 优化阶段 (Optimization)：最小化“置信度差距”
    - $\mathcal{L}_{GOLD}(\theta) = \mathbb{E}_{x \sim \mathcal{D}, \hat{y} \sim \pi_\theta} \left[ D_{seq}(\pi_\theta, \pi_\phi) \right]$
- 在具体操作层面，这通常简化为让学生去匹配老师对该序列的似然估计。
    - 如果采用类似于 JSD (Jensen-Shannon Divergence) 或单纯的 Logit Regression 思想，损失函数可以写成：
        - $\mathcal{L}(\theta) \approx \left\| \log P_\theta(\hat{y}|x) - \log P_\phi(\hat{y}|x) \right\|^2$
    - 或者在强化学习框架下（如 PPO 形式），我们将老师的概率视作 Reward（奖励）：
        - $R(x, \hat{y}) = \log P_\phi(\hat{y}|x) - \beta \log P_\theta(\hat{y}|x)$